# Wald Rank Tests for All Example Models
**From:** Zhang, Wu, Shi, & Savalei — *Model Identification Made Clear*

Each model defines:
- `EqVec` = vech of the model-implied covariance matrix (symbolic)
- `params` = list of free parameters
- `Jcb` = Jacobian of EqVec w.r.t. params
- `rank` = MatrixRank(Jcb)

**Convention:** first loading on each factor fixed to 1 unless otherwise noted.

In [3]:
from sympy import symbols, Matrix, MatrixSymbol, zeros, eye, ones, diag
from sympy import Symbol, Function, sqrt, Rational


def vech(M):
    """Extract lower-triangular elements (column-major) of symmetric matrix."""
    n = M.rows
    elems = []
    for j in range(n):
        for i in range(j, n):
            elems.append(M[i, j])
    return elems


def build_model(Lambda, Phi, Psi):
    """Compute Sigma = Lambda @ Phi @ Lambda^T + Psi, return vech(Sigma)."""
    Sigma = Lambda * Phi * Lambda.T + Psi
    return vech(Sigma)


def wald_rank_test(EqVec, params, label=""):
    """Compute Jacobian and its rank. Print results."""
    EqMat = Matrix(EqVec)
    Jcb = EqMat.jacobian(params)
    rank = Jcb.rank()
    q = len(params)
    status = "Identified (rank = q)" if rank == q else f"Rank-deficient (rank < q)"
    print(f"{label}")
    print(f"  rank = {rank}, q = {q}  -->  {status}")
    print()
    return rank, q

## Table 1: Just-Identified, Over-Identified, and Unidentified

In [4]:
# --- Model (a): One-factor, 3 indicators, lambda1 = 1 ---
phi, lam2, lam3 = symbols('phi lambda2 lambda3', nonzero=True)
psi1, psi2, psi3 = symbols('psi1 psi2 psi3')

EqVec = [
    phi + psi1,
    phi*lam2,
    phi*lam3,
    phi*lam2**2 + psi2,
    phi*lam2*lam3,
    phi*lam3**2 + psi3,
]
params = [phi, lam2, lam3, psi1, psi2, psi3]
wald_rank_test(EqVec, params, "Table 1 Model (a): 1-factor, 3 indicators, lam1=1")

Table 1 Model (a): 1-factor, 3 indicators, lam1=1
  rank = 6, q = 6  -->  Identified (rank = q)



(6, 6)

In [5]:
# --- Model (b): One-factor, 3 indicators, lambda1 = 1, lambda2 = 1 ---
phi, lam3 = symbols('phi lambda3', nonzero=True)
psi1, psi2, psi3 = symbols('psi1 psi2 psi3')

EqVec = [
    phi + psi1,
    phi,
    phi*lam3,
    phi + psi2,
    phi*lam3,
    phi*lam3**2 + psi3,
]
params = [phi, lam3, psi1, psi2, psi3]
wald_rank_test(EqVec, params, "Table 1 Model (b): 1-factor, 3 indicators, lam1=lam2=1")

Table 1 Model (b): 1-factor, 3 indicators, lam1=lam2=1
  rank = 5, q = 5  -->  Identified (rank = q)



(5, 5)

In [6]:
# --- Model (c): Two uncorrelated factors, 2 indicators each ---
phi1, phi2 = symbols('phi1 phi2', nonzero=True)
lam2, lam4 = symbols('lambda2 lambda4', nonzero=True)
psi1, psi2, psi3, psi4 = symbols('psi1 psi2 psi3 psi4')

EqVec = [
    phi1 + psi1,          # s11
    phi1*lam2,            # s21
    0,                    # s31
    0,                    # s41
    phi1*lam2**2 + psi2,  # s22
    0,                    # s32
    0,                    # s42
    phi2 + psi3,          # s33
    phi2*lam4,            # s43
    phi2*lam4**2 + psi4,  # s44
]
params = [phi1, phi2, lam2, lam4, psi1, psi2, psi3, psi4]
wald_rank_test(EqVec, params, "Table 1 Model (c): 2 uncorrelated factors, 2 indicators each")

Table 1 Model (c): 2 uncorrelated factors, 2 indicators each
  rank = 6, q = 8  -->  Rank-deficient (rank < q)



(6, 8)

In [7]:
# --- Model (d): Two correlated factors, 2 indicators each ---
phi1, phi2, phi12 = symbols('phi1 phi2 phi12', nonzero=True)
lam2, lam4 = symbols('lambda2 lambda4', nonzero=True)
psi1, psi2, psi3, psi4 = symbols('psi1 psi2 psi3 psi4')

EqVec = [
    phi1 + psi1,                # s11
    phi1*lam2,                  # s21
    phi12,                      # s31
    phi12*lam4,                 # s41
    phi1*lam2**2 + psi2,        # s22
    phi12*lam2,                 # s32
    phi12*lam2*lam4,            # s42
    phi2 + psi3,                # s33
    phi2*lam4,                  # s43
    phi2*lam4**2 + psi4,        # s44
]
params = [phi1, phi2, phi12, lam2, lam4, psi1, psi2, psi3, psi4]
wald_rank_test(EqVec, params, "Table 1 Model (d): 2 correlated factors, 2 indicators each")

Table 1 Model (d): 2 correlated factors, 2 indicators each
  rank = 9, q = 9  -->  Identified (rank = q)



(9, 9)

## Table 2: One-Factor Models with Correlated Errors

In [8]:
# --- Model (a): 1-factor, 5 indicators, x4 has CE with ALL others ---
phi = symbols('phi', nonzero=True)
lam2, lam3, lam4, lam5 = symbols('lambda2 lambda3 lambda4 lambda5', nonzero=True)
psi1, psi2, psi3, psi4, psi5 = symbols('psi1 psi2 psi3 psi4 psi5')
psi14, psi24, psi34, psi45 = symbols('psi14 psi24 psi34 psi45')

EqVec = [
    phi + psi1,                    # s11
    phi*lam2,                      # s21
    phi*lam3,                      # s31
    phi*lam4 + psi14,              # s41
    phi*lam5,                      # s51
    phi*lam2**2 + psi2,            # s22
    phi*lam2*lam3,                 # s32
    phi*lam2*lam4 + psi24,         # s42
    phi*lam2*lam5,                 # s52
    phi*lam3**2 + psi3,            # s33
    phi*lam3*lam4 + psi34,         # s43
    phi*lam3*lam5,                 # s53
    phi*lam4**2 + psi4,            # s44
    phi*lam4*lam5 + psi45,         # s54
    phi*lam5**2 + psi5,            # s55
]
params = [phi, lam2, lam3, lam4, lam5,
          psi1, psi2, psi3, psi4, psi5,
          psi14, psi24, psi34, psi45]
wald_rank_test(EqVec, params,
    "Table 2 Model (a): 1-factor, 5 indicators, x4 CE with all others")

Table 2 Model (a): 1-factor, 5 indicators, x4 CE with all others
  rank = 13, q = 14  -->  Rank-deficient (rank < q)



(13, 14)

In [18]:
# --- Model (b): 1-factor, 5 indicators, CE satisfying Theorem 3 ---
phi = symbols('phi', nonzero=True)
lam2, lam3, lam4, lam5 = symbols('lambda2 lambda3 lambda4 lambda5', nonzero=True)
psi1, psi2, psi3, psi4, psi5 = symbols('psi1 psi2 psi3 psi4 psi5')
psi14, psi34, psi15, psi25, psi45 = symbols('psi14 psi34 psi15 psi25 psi45')

EqVec = [
    phi + psi1,                 # s11
    phi*lam2,                   # s21
    phi*lam3,                   # s31
    phi*lam4 + psi14,           # s41
    phi*lam5 + psi15,           # s51
    phi*lam2**2 + psi2,         # s22
    phi*lam2*lam3,              # s32
    phi*lam2*lam4,              # s42
    phi*lam2*lam5 + psi25,      # s52
    phi*lam3**2 + psi3,         # s33
    phi*lam3*lam4 + psi34,      # s43
    phi*lam3*lam5,              # s53
    phi*lam4**2 + psi4,         # s44
    phi*lam4*lam5 + psi45,      # s54
    phi*lam5**2 + psi5,         # s55
]

params = [phi, lam2, lam3, lam4, lam5,
          psi1, psi2, psi3, psi4, psi5,
          psi14, psi34, psi15, psi25, psi45]

wald_rank_test(EqVec, params,
               "Table 2 Model (b): 1-factor, 5 indicators, CE with (3,4) instead of (2,4)")

Table 2 Model (b): 1-factor, 5 indicators, CE with (3,4) instead of (2,4)
  rank = 15, q = 15  -->  Identified (rank = q)



(15, 15)

In [10]:
# --- Model (c): CTCU, 6 indicators, 2 method effects ---
phi = symbols('phi', nonzero=True)
lam2, lam3, lam4, lam5, lam6 = symbols(
    'lambda2 lambda3 lambda4 lambda5 lambda6', nonzero=True)
psi1, psi2, psi3, psi4, psi5, psi6 = symbols('psi1 psi2 psi3 psi4 psi5 psi6')
psi12, psi13, psi23 = symbols('psi12 psi13 psi23')
psi45, psi46, psi56 = symbols('psi45 psi46 psi56')

EqVec = [
    phi + psi1,                    # s11
    phi*lam2 + psi12,              # s21
    phi*lam3 + psi13,              # s31
    phi*lam4,                      # s41
    phi*lam5,                      # s51
    phi*lam6,                      # s61
    phi*lam2**2 + psi2,            # s22
    phi*lam2*lam3 + psi23,         # s32
    phi*lam2*lam4,                 # s42
    phi*lam2*lam5,                 # s52
    phi*lam2*lam6,                 # s62
    phi*lam3**2 + psi3,            # s33
    phi*lam3*lam4,                 # s43
    phi*lam3*lam5,                 # s53
    phi*lam3*lam6,                 # s63
    phi*lam4**2 + psi4,            # s44
    phi*lam4*lam5 + psi45,         # s54
    phi*lam4*lam6 + psi46,         # s64
    phi*lam5**2 + psi5,            # s55
    phi*lam5*lam6 + psi56,         # s65
    phi*lam6**2 + psi6,            # s66
]
params = [phi, lam2, lam3, lam4, lam5, lam6,
          psi1, psi2, psi3, psi4, psi5, psi6,
          psi12, psi13, psi23, psi45, psi46, psi56]
wald_rank_test(EqVec, params,
    "Table 2 Model (c): CTCU, 6 indicators, CE within {x1,x2,x3} and {x4,x5,x6}")

Table 2 Model (c): CTCU, 6 indicators, CE within {x1,x2,x3} and {x4,x5,x6}
  rank = 17, q = 18  -->  Rank-deficient (rank < q)



(17, 18)

In [11]:
# --- Model (d): Same as (c) but delete psi12 ---
phi = symbols('phi', nonzero=True)
lam2, lam3, lam4, lam5, lam6 = symbols(
    'lambda2 lambda3 lambda4 lambda5 lambda6', nonzero=True)
psi1, psi2, psi3, psi4, psi5, psi6 = symbols('psi1 psi2 psi3 psi4 psi5 psi6')
psi13, psi23 = symbols('psi13 psi23')
psi45, psi46, psi56 = symbols('psi45 psi46 psi56')

EqVec = [
    phi + psi1,                    # s11
    phi*lam2,                      # s21  (psi12 deleted)
    phi*lam3 + psi13,              # s31
    phi*lam4,                      # s41
    phi*lam5,                      # s51
    phi*lam6,                      # s61
    phi*lam2**2 + psi2,            # s22
    phi*lam2*lam3 + psi23,         # s32
    phi*lam2*lam4,                 # s42
    phi*lam2*lam5,                 # s52
    phi*lam2*lam6,                 # s62
    phi*lam3**2 + psi3,            # s33
    phi*lam3*lam4,                 # s43
    phi*lam3*lam5,                 # s53
    phi*lam3*lam6,                 # s63
    phi*lam4**2 + psi4,            # s44
    phi*lam4*lam5 + psi45,         # s54
    phi*lam4*lam6 + psi46,         # s64
    phi*lam5**2 + psi5,            # s55
    phi*lam5*lam6 + psi56,         # s65
    phi*lam6**2 + psi6,            # s66
]
params = [phi, lam2, lam3, lam4, lam5, lam6,
          psi1, psi2, psi3, psi4, psi5, psi6,
          psi13, psi23, psi45, psi46, psi56]
wald_rank_test(EqVec, params,
    "Table 2 Model (d): Same as (c) minus psi12")

Table 2 Model (d): Same as (c) minus psi12
  rank = 17, q = 17  -->  Identified (rank = q)



(17, 17)

In [12]:
# --- Model (e): 1-factor, 6 indicators, CE only among {x4,x5,x6} ---
phi = symbols('phi', nonzero=True)
lam2, lam3, lam4, lam5, lam6 = symbols(
    'lambda2 lambda3 lambda4 lambda5 lambda6', nonzero=True)
psi1, psi2, psi3, psi4, psi5, psi6 = symbols('psi1 psi2 psi3 psi4 psi5 psi6')
psi45, psi46, psi56 = symbols('psi45 psi46 psi56')

EqVec = [
    phi + psi1,                    # s11
    phi*lam2,                      # s21
    phi*lam3,                      # s31
    phi*lam4,                      # s41
    phi*lam5,                      # s51
    phi*lam6,                      # s61
    phi*lam2**2 + psi2,            # s22
    phi*lam2*lam3,                 # s32
    phi*lam2*lam4,                 # s42
    phi*lam2*lam5,                 # s52
    phi*lam2*lam6,                 # s62
    phi*lam3**2 + psi3,            # s33
    phi*lam3*lam4,                 # s43
    phi*lam3*lam5,                 # s53
    phi*lam3*lam6,                 # s63
    phi*lam4**2 + psi4,            # s44
    phi*lam4*lam5 + psi45,         # s54
    phi*lam4*lam6 + psi46,         # s64
    phi*lam5**2 + psi5,            # s55
    phi*lam5*lam6 + psi56,         # s65
    phi*lam6**2 + psi6,            # s66
]
params = [phi, lam2, lam3, lam4, lam5, lam6,
          psi1, psi2, psi3, psi4, psi5, psi6,
          psi45, psi46, psi56]
wald_rank_test(EqVec, params,
    "Table 2 Model (e): 1-factor, 6 indicators, CE only among {x4,x5,x6}")

Table 2 Model (e): 1-factor, 6 indicators, CE only among {x4,x5,x6}
  rank = 15, q = 15  -->  Identified (rank = q)



(15, 15)

## Table 3: Correlated-Factor, Correlated Errors, No Cross-Loadings
8 indicators, 2 factors: F1 → x1(=1), x2, x3, x4; F2 → x5(=1), x6, x7, x8

In [13]:
# Common symbols for Table 3
lam2, lam3, lam4 = symbols('lambda2 lambda3 lambda4', nonzero=True)
lam6, lam7, lam8 = symbols('lambda6 lambda7 lambda8', nonzero=True)
phi1, phi2, phi12 = symbols('phi1 phi2 phi12', nonzero=True)
psi = symbols('psi1:9')  # psi1 through psi8


def table3_base_sigma(ce_dict):
    """
    Build vech(Sigma) for 2-factor model (8 indicators, no cross-loadings).
    F1 -> x1(=1), x2, x3, x4
    F2 -> x5(=1), x6, x7, x8
    ce_dict: {(i,j): symbol} for correlated errors (1-indexed)
    """
    L1 = [1, lam2, lam3, lam4, 0, 0, 0, 0]
    L2 = [0, 0, 0, 0, 1, lam6, lam7, lam8]

    eqvec = []
    for j in range(8):
        for i in range(j, 8):
            sij = L1[i]*phi1*L1[j] + L2[i]*phi2*L2[j] \
                + (L1[i]*L2[j] + L2[i]*L1[j])*phi12
            if i == j:
                sij += psi[i]
            else:
                key = (j+1, i+1)
                if key in ce_dict:
                    sij += ce_dict[key]
            eqvec.append(sij)
    return eqvec

In [19]:
# --- Model (a): Within-factor CE: psi14, psi24, psi68, psi78 ---
p14s, p24s, p68s, p78s = symbols('psi14 psi24 psi68 psi78')

ce_a = {
    (1,4): p14s,
    (2,4): p24s,
    (6,8): p68s,
    (7,8): p78s
}

EqVec = table3_base_sigma(ce_a)

params = [lam2, lam3, lam4, lam6, lam7, lam8,
          phi1, phi2, phi12,
          *psi, p14s, p24s, p68s, p78s]

wald_rank_test(EqVec, params,
    "Table 3 Model (a): within-factor CE (14, 24, 68, 78)")

Table 3 Model (a): within-factor CE (14, 24, 68, 78)
  rank = 21, q = 21  -->  Identified (rank = q)



(21, 21)

In [20]:
# --- Model (b): Model (a) + all cross-factor CE except (1,5) ---
# Within-factor CE from updated Model (a)
p14s, p24s, p68s, p78s = symbols('psi14 psi24 psi68 psi78')
ce_a = {
    (1,4): p14s,
    (2,4): p24s,
    (6,8): p68s,
    (7,8): p78s
}

# Build all cross-factor CE except (1,5)
cross_ce_syms = {}
cross_ce_params = []

for i in range(1, 5):      # x1–x4 (Factor 1)
    for j in range(5, 9):  # x5–x8 (Factor 2)
        if (i, j) == (1, 5):
            continue
        s = Symbol(f'psi{i}{j}')
        cross_ce_syms[(i, j)] = s
        cross_ce_params.append(s)

# Combine CE
ce_b = {**ce_a, **cross_ce_syms}

# Build covariance
EqVec = table3_base_sigma(ce_b)

params = [lam2, lam3, lam4, lam6, lam7, lam8,
          phi1, phi2, phi12,
          *psi, p14s, p24s, p68s, p78s,
          *cross_ce_params]

wald_rank_test(EqVec, params,
    "Table 3 Model (b): within CE (14,24,68,78) + all cross CE except (1,5)")

Table 3 Model (b): within CE (14,24,68,78) + all cross CE except (1,5)
  rank = 36, q = 36  -->  Identified (rank = q)



(36, 36)

In [21]:
# --- Model (c): Fix phi12=0, keep updated within-factor CE, add all cross-factor CE ---
def table3c_sigma(ce_dict):
    L1 = [1, lam2, lam3, lam4, 0, 0, 0, 0]
    L2 = [0, 0, 0, 0, 1, lam6, lam7, lam8]
    eqvec = []
    for j in range(8):
        for i in range(j, 8):
            sij = L1[i]*phi1*L1[j] + L2[i]*phi2*L2[j]
            # phi12 = 0, so cross terms vanish
            if i == j:
                sij += psi[i]
            else:
                key = (j+1, i+1)
                if key in ce_dict:
                    sij += ce_dict[key]
            eqvec.append(sij)
    return eqvec

# Updated within-factor CE
p14s, p24s, p68s, p78s = symbols('psi14 psi24 psi68 psi78')

# All 16 cross-factor CE
cross_ce_all_syms = {}
cross_ce_all_params = []
for i in range(1, 5):
    for j in range(5, 9):
        s = Symbol(f'psi{i}{j}')
        cross_ce_all_syms[(i, j)] = s
        cross_ce_all_params.append(s)

ce_c = {
    **{(1,4): p14s, (2,4): p24s, (6,8): p68s, (7,8): p78s},
    **cross_ce_all_syms
}

EqVec = table3c_sigma(ce_c)

params = [lam2, lam3, lam4, lam6, lam7, lam8,
          phi1, phi2,   # no phi12
          *psi, p14s, p24s, p68s, p78s,
          *cross_ce_all_params]

wald_rank_test(EqVec, params,
    "Table 3 Model (c): phi12=0, within CE (14,24,68,78), all 16 cross-factor CE")

Table 3 Model (c): phi12=0, within CE (14,24,68,78), all 16 cross-factor CE
  rank = 36, q = 36  -->  Identified (rank = q)



(36, 36)

## Table 4: Correlated-Factor with Cross-Loadings & CE
8 indicators, 2 factors

In [26]:
def table4_sigma(Lambda_list, phi_mat, psi_diag, ce_dict):
    """
    Lambda_list: list of (loading_F1, loading_F2) per indicator
    phi_mat: [[phi11, phi12], [phi12, phi22]]
    psi_diag: list of error variance symbols
    ce_dict: {(i,j): symbol} 1-indexed
    """
    p = len(Lambda_list)
    eqvec = []
    for j in range(p):
        for i in range(j, p):
            li1, li2 = Lambda_list[i]
            lj1, lj2 = Lambda_list[j]
            sij = (li1*lj1*phi_mat[0][0] + li1*lj2*phi_mat[0][1]
                   + li2*lj1*phi_mat[1][0] + li2*lj2*phi_mat[1][1])
            if i == j:
                sij += psi_diag[i]
            else:
                key = (j+1, i+1)
                if key in ce_dict:
                    sij += ce_dict[key]
            eqvec.append(sij)
    return eqvec

Table 4 Model (a): x1 on F1 only, x2 on F2 only, x3-x8 all cross-load
  rank = 23, q = 23  -->  Identified (rank = q)



(23, 23)

In [27]:
# --- Model (a): Cross-loading model ---
phi1, phi2, phi12 = symbols('phi1 phi2 phi12', nonzero=True)
l31, l32 = symbols('lambda31 lambda32', nonzero=True)
l41, l42 = symbols('lambda41 lambda42', nonzero=True)
l51, l52 = symbols('lambda51 lambda52', nonzero=True)
l61, l62 = symbols('lambda61 lambda62', nonzero=True)
l71, l72 = symbols('lambda71 lambda72', nonzero=True)
l81, l82 = symbols('lambda81 lambda82', nonzero=True)
psi_t4 = symbols('psi1:9')

Lambda_4a = [
    (1, 0),         # x1: F1 only
    (0, 1),         # x2: F2 only
    (l31, l32),     # x3: cross-load
    (l41, l42),     # x4: cross-load
    (l51, l52),     # x5: cross-load
    (l61, l62),     # x6: cross-load
    (l71, l72),     # x7: cross-load
    (l81, l82),     # x8: cross-load
]

phi_4a = [[phi1, phi12], [phi12, phi2]]

EqVec = table4_sigma(Lambda_4a, phi_4a, psi_t4, {})

params = [l31, l32, l41, l42, l51, l52, l61, l62, l71, l72, l81, l82,
          phi1, phi2, phi12, *psi_t4]

wald_rank_test(EqVec, params,
    "Table 4 Model (a): x1 on F1 only, x2 on F2 only, x3-x8 all cross-load")

Table 4 Model (a): x1 on F1 only, x2 on F2 only, x3-x8 all cross-load
  rank = 23, q = 23  -->  Identified (rank = q)



(23, 23)

In [28]:
# --- Model (b): same as updated Model (a), but phi12=0 and x2 also loads on F1 ---
l21 = symbols('lambda21', nonzero=True)

Lambda_4b = [
    (1, 0),          # x1: F1 only
    (l21, 1),        # x2: F2 marker + extra loading on F1
    (l31, l32),      # x3: cross-load
    (l41, l42),      # x4: cross-load
    (l51, l52),      # x5: cross-load
    (l61, l62),      # x6: cross-load
    (l71, l72),      # x7: cross-load
    (l81, l82),      # x8: cross-load
]

phi_4b = [[phi1, 0], [0, phi2]]

EqVec = table4_sigma(Lambda_4b, phi_4b, psi_t4, {})

params = [l21,
          l31, l32, l41, l42, l51, l52, l61, l62, l71, l72, l81, l82,
          phi1, phi2, *psi_t4]

wald_rank_test(EqVec, params,
    "Table 4 Model (b): phi12=0, x2 also loads on F1, x3-x8 cross-load")

Table 4 Model (b): phi12=0, x2 also loads on F1, x3-x8 cross-load
  rank = 23, q = 23  -->  Identified (rank = q)



(23, 23)

In [29]:
# --- Model (c): Cross-loading + correlated errors (Theorem 7) ---
# Sets:
# x1 = (x1, x2)
# x2 = (x3, x4)
# x3 = (x5, x6)
# x4 = (x7, x8)
#
# CE within x1, x2, x3: (1,2), (3,4), (5,6)
# x7 CE with all except x1: (3,7), (4,7), (5,7), (6,7), (7,8)
# x8 CE with all except x3: (1,8), (2,8), (3,8), (4,8), (7,8)
# Unique CE set:
# (1,2), (3,4), (5,6), (3,7), (4,7), (5,7), (6,7), (7,8), (1,8), (2,8), (3,8), (4,8)

ce_4c_syms = {}
ce_4c_params = []

for pair in [(1,2), (3,4), (5,6),
             (3,7), (4,7), (5,7), (6,7), (7,8),
             (1,8), (2,8), (3,8), (4,8)]:
    s = Symbol(f'psi{pair[0]}{pair[1]}')
    ce_4c_syms[pair] = s
    ce_4c_params.append(s)

EqVec = table4_sigma(Lambda_4a, phi_4a, psi_t4, ce_4c_syms)

params = [l31, l32, l41, l42, l51, l52, l61, l62, l71, l72, l81, l82,
          phi1, phi2, phi12, *psi_t4, *ce_4c_params]

wald_rank_test(EqVec, params,
    "Table 4 Model (c): Cross-loading + CE matching Theorem 7 description")

Table 4 Model (c): Cross-loading + CE matching Theorem 7 description
  rank = 35, q = 35  -->  Identified (rank = q)



(35, 35)

## Table 5: CTCU Models with Multiple Factors

In [30]:
lam2_5, lam3_5 = symbols('lambda2 lambda3', nonzero=True)
lam5_5, lam6_5 = symbols('lambda5 lambda6', nonzero=True)
lam8_5, lam9_5 = symbols('lambda8 lambda9', nonzero=True)
phi1_5, phi2_5, phi3_5 = symbols('phi1 phi2 phi3', nonzero=True)
phi12_5, phi13_5, phi23_5 = symbols('phi12 phi13 phi23')
psi_5 = symbols('psi1:10')  # psi1 through psi9

p14_5, p17_5, p47_5 = symbols('psi14 psi17 psi47')
p25_5, p28_5, p58_5 = symbols('psi25 psi28 psi58')
p36_5, p39_5, p69_5 = symbols('psi36 psi39 psi69')

def table5_sigma(Lambda3, phi3, psi9, ce_dict):
    """3-factor model, 9 indicators."""
    p = 9
    eqvec = []
    for j in range(p):
        for i in range(j, p):
            sij = 0
            for k in range(3):
                for l in range(3):
                    sij += Lambda3[i][k] * phi3[k][l] * Lambda3[j][l]
            if i == j:
                sij += psi9[i]
            else:
                key = (j+1, i+1)
                if key in ce_dict:
                    sij += ce_dict[key]
            eqvec.append(sij)
    return eqvec

Lambda_5a = [
    [1, 0, 0],          # x1: F1 marker
    [lam2_5, 0, 0],     # x2: F1
    [lam3_5, 0, 0],     # x3: F1
    [0, 1, 0],          # x4: F2 marker
    [0, lam5_5, 0],     # x5: F2
    [0, lam6_5, 0],     # x6: F2
    [0, 0, 1],          # x7: F3 marker
    [0, 0, lam8_5],     # x8: F3
    [0, 0, lam9_5],     # x9: F3
]
phi_5a = [
    [phi1_5, phi12_5, phi13_5],
    [phi12_5, phi2_5, phi23_5],
    [phi13_5, phi23_5, phi3_5],
]

In [31]:
# --- Model (a): 3 correlated factors, NO method CE ---
EqVec = table5_sigma(Lambda_5a, phi_5a, psi_5, {})
params = [lam2_5, lam3_5, lam5_5, lam6_5, lam8_5, lam9_5,
          phi1_5, phi2_5, phi3_5, phi12_5, phi13_5, phi23_5,
          *psi_5]
wald_rank_test(EqVec, params,
    "Table 5 Model (a): 3 correlated factors, 9 indicators, NO CE")

Table 5 Model (a): 3 correlated factors, 9 indicators, NO CE
  rank = 21, q = 21  -->  Identified (rank = q)



(21, 21)

In [32]:
# --- Model (a) WITH method CE ---
ce_5a = {
    (1,4): p14_5, (1,7): p17_5, (4,7): p47_5,
    (2,5): p25_5, (2,8): p28_5, (5,8): p58_5,
    (3,6): p36_5, (3,9): p39_5, (6,9): p69_5,
}
EqVec = table5_sigma(Lambda_5a, phi_5a, psi_5, ce_5a)
params_ce = [lam2_5, lam3_5, lam5_5, lam6_5, lam8_5, lam9_5,
             phi1_5, phi2_5, phi3_5, phi12_5, phi13_5, phi23_5,
             *psi_5,
             p14_5, p17_5, p47_5, p25_5, p28_5, p58_5,
             p36_5, p39_5, p69_5]
wald_rank_test(EqVec, params_ce,
    "Table 5 Model (a) WITH CE: 3 correlated factors + method CE")

Table 5 Model (a) WITH CE: 3 correlated factors + method CE
  rank = 30, q = 30  -->  Identified (rank = q)



(30, 30)

In [33]:
# --- Model (b): 2 factors, 8 indicators, CTCU with added CE ---

lam2_5b, lam3_5b, lam4_5b = symbols('lambda2 lambda3 lambda4', nonzero=True)
lam6_5b, lam7_5b, lam8_5b = symbols('lambda6 lambda7 lambda8', nonzero=True)
phi1_5b, phi2_5b, phi12_5b = symbols('phi1 phi2 phi12', nonzero=True)
psi_5b = symbols('psi1:9')

# original CE
p12_5b = Symbol('psi12')
p34_5b = Symbol('psi34')
p56_5b = Symbol('psi56')
p78_5b = Symbol('psi78')

# new CE you want
p37_5b = Symbol('psi37')
p38_5b = Symbol('psi38')

ce_5b = {
    (1,2): p12_5b,
    (3,4): p34_5b,
    (5,6): p56_5b,
    (7,8): p78_5b,
    (3,7): p37_5b,   # NEW
    (3,8): p38_5b,   # NEW
}

# reuse structure
lam2_old, lam3_old, lam4_old = lam2, lam3, lam4
lam6_old, lam7_old, lam8_old = lam6, lam7, lam8

lam2 = lam2_5b; lam3 = lam3_5b; lam4 = lam4_5b
lam6 = lam6_5b; lam7 = lam7_5b; lam8 = lam8_5b
phi1 = phi1_5b; phi2 = phi2_5b; phi12 = phi12_5b
psi = psi_5b

EqVec = table3_base_sigma(ce_5b)

params = [lam2, lam3, lam4, lam6, lam7, lam8,
          phi1, phi2, phi12,
          *psi,
          p12_5b, p34_5b, p56_5b, p78_5b,
          p37_5b, p38_5b]

wald_rank_test(EqVec, params,
    "Table 5 Model (b): + CE (3,7), (3,8)")

Table 5 Model (b): + CE (3,7), (3,8)
  rank = 23, q = 23  -->  Identified (rank = q)



(23, 23)